# M3 historical CLV 엣지 배분 그래프 — Dunnhumby 1시드 파일럿

- 학습: DAY 1~683
- 평가: DAY 684~690 신규상품 추천
- seed 42, 100 epoch 고정
- validation·early stopping·최종 test·holdout 없음
- 한 시드 결과이므로 분산·신뢰구간·유의성·일반화를 주장하지 않음


In [ ]:
from google.colab import drive
from pathlib import Path
import importlib
import subprocess
import sys

drive.mount('/content/drive')
REVIEWED_SHA = '6885c8dd9829fd80ae904be435c4daf15ce05974'
REPO = Path('/content/clv-m2-lightgcn-runner')
if not (REPO / '.git').exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/jung-un/clv-m2-lightgcn-runner.git',
        str(REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(
    ['git', '-C', str(REPO), 'checkout', '--detach', REVIEWED_SHA],
    check=True,
)
actual_sha = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in list(sys.modules):
    if module_name.startswith(('clv_', 'lightgcn_clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print('검토된 실행 소스:', REVIEWED_SHA)


In [ ]:
import json
from lightgcn_clv_m3_edge_allocation_backtest import (
    configure_m3_edge_allocation_backtest,
    preflight_summary,
)

OUT_DIR = (
    '/content/drive/MyDrive/논문/data/'
    'results_m3_clv_edge_allocation_historical_dunnhumby'
)
cfg = configure_m3_edge_allocation_backtest(out_dir=OUT_DIR)
preflight = preflight_summary(cfg)
print(json.dumps(preflight, ensure_ascii=False, indent=2))
split = preflight['historical_development_split']
assert preflight['seed'] == 42
assert preflight['epochs'] == 100
assert split['train_end_inclusive'] == 683
assert split['evaluation_start_inclusive'] == 684
assert split['evaluation_end_inclusive'] == 690
assert split['final_test_constructed'] is False
assert split['holdout_constructed'] is False


In [ ]:
from lightgcn_clv_m3_edge_allocation_backtest import (
    run_m3_edge_allocation_backtest,
)

result_df = run_m3_edge_allocation_backtest(cfg)


In [ ]:
from IPython.display import display

public_columns = [
    'model_id', 'role',
    'recall@10', 'ndcg@10',
    'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'mean_recommended_price_percentile@10',
    'coverage@10', 'n_distinct@10',
    'exposure_entropy@10', 'eff_catalog@10',
    'top10_share@10', 'top100_share@10',
]
display(result_df[[c for c in public_columns if c in result_df.columns]])
print('M3 - M1 비교:')
display(result_df.attrs['comparison'])
print('사전 고정 파일럿 판정:')
print(json.dumps(result_df.attrs['pilot_decision'], ensure_ascii=False, indent=2))
print('결과 파일:', result_df.attrs['result_paths'])
